# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [1]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Eliminamos columnas no útiles
df = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])

# Convertimos target a numérico si es booleano
df["Transported"] = df["Transported"].astype(int)

X = df.drop("Transported", axis=1)
y = df["Transported"]

In [11]:
#oNE HOT ENCODING
X = pd.get_dummies(X, drop_first=True)

**Perform Train Test Split**

In [12]:
#train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [13]:
#standxardscaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [16]:
from sklearn.ensemble import BaggingClassifier

In [17]:
bootstrap=True   # Bagging
bootstrap=False  # Pasting

In [18]:
#Preparacion de datos
import pandas as pd
from sklearn.model_selection import train_test_split

# Cargar datos
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")

# Limpieza básica
df = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])
df["Transported"] = df["Transported"].astype(int)

X = pd.get_dummies(df.drop("Transported", axis=1), drop_first=True)
y = df["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
# BAGGING
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

base_model = DecisionTreeClassifier()

bagging = BaggingClassifier(
    estimator=base_model,
    n_estimators=100,
    bootstrap=True,      # 👈 Bagging
    random_state=42,
    n_jobs=-1
)

bagging.fit(X_train, y_train)

y_pred = bagging.predict(X_test)

print("Accuracy Bagging:", accuracy_score(y_test, y_pred))

Accuracy Bagging: 0.7688326624496837


In [20]:
#Pasting

pasting = BaggingClassifier(
    estimator=base_model,
    n_estimators=100,
    bootstrap=False,     # 👈 Pasting
    random_state=42,
    n_jobs=-1
)

pasting.fit(X_train, y_train)

y_pred_paste = pasting.predict(X_test)

print("Accuracy Pasting:", accuracy_score(y_test, y_pred_paste))

Accuracy Pasting: 0.7211040828062104


- Random Forests

In [21]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print("Accuracy Random Forest:", rf.score(X_test, y_test))

Accuracy Random Forest: 0.7694077055779184


- Gradient Boosting

In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split

spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")

# Eliminamos columnas que no aportan
df = spaceship.drop(columns=["PassengerId", "Name", "Cabin"])

# Convertimos target
df["Transported"] = df["Transported"].astype(int)

X = df.drop("Transported", axis=1)
y = df["Transported"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [25]:
#separara columnas
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "bool"]).columns
print("Accuracy Gradient Boosting:", accuracy_score(y_test, y_pred))

Accuracy Gradient Boosting: 0.7688326624496837


In [26]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Pipeline para numéricas
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Pipeline para categóricas
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

In [27]:
#pipeline con gradient boosting

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7809085681426107


- Adaptive Boosting

In [30]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

ada_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1),
        n_estimators=200,
        learning_rate=0.5,
        random_state=42
    ))
])

ada_pipeline.fit(X_train, y_train)

y_pred = ada_pipeline.predict(X_test)

print("Accuracy AdaBoost:", accuracy_score(y_test, y_pred))

Accuracy AdaBoost: 0.7584818861414606


Which model is the best and why?

In [ ]:
para este tipo de dataset, la mejor opción sería Gradient Boosting, ya que nos da el mejor Accuracy, no obstante, AdaBoost también es una buena opción y podría ser preferible si queremos un modelo más interpretable. Random Forest es una opción sólida pero no tan precisa como los anteriores en este caso.